# BERT-family SQLi Fine-tuning — Colab Wrapper

Thin wrapper around `train.py`. All training logic lives there; this notebook just:
1. Mounts Google Drive and installs dependencies
2. Clones the repo (or pulls latest)
3. Selects a named training run (`RUN`) and overrides key config values
4. Calls `t.main()`
5. Downloads the run log

**Three training runs make up the thesis experiment design (D3.7) — run this
notebook once per `RUN` value:**

| `RUN` | Model | Data | Role |
|---|---|---|---|
| `codebert-sqli` | CodeBERT | full corpus | the product (headline model) |
| `bert-sqli-500k` | BERT | 500K rows/split | PP6 control |
| `codebert-sqli-500k` | CodeBERT | 500K rows/split | PP6 control (fair partner) |

**Before running:**
- Set runtime to GPU: Runtime → Change runtime type → Hardware accelerator → **GPU (T4)**
- Upload `compiled_dataset.csv` to `My Drive/mestrado/compiled_dataset.csv`

In [1]:
# ── Cell 1: Environment setup ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Colab already ships a CUDA-enabled torch — do NOT include torch here.
# pip install torch pulls the CPU-only wheel and overwrites Colab's CUDA build.
!pip install -q pandas scikit-learn imbalanced-learn transformers matplotlib seaborn psutil codecarbon

import os
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

REPO_DIR = '/content/mestrado'
if not os.path.exists(REPO_DIR):
    !git clone -b sliding-window https://github.com/l-a-motta/mestrado-sqli-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch: 2.10.0+cu128 | CUDA available: True
Already up to date.
Working directory: /content/mestrado


In [2]:
# ── Cell 3: Flush GPU and RAM before training ─────────────────────────────────
import gc
import torch
import psutil

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    props = torch.cuda.get_device_properties(0)
    free  = props.total_memory - torch.cuda.memory_allocated(0)
    print(f'GPU  : {props.name}  total={props.total_memory/1e9:.1f} GB  free={free/1e9:.1f} GB')

ram = psutil.virtual_memory()
print(f'RAM  : total={ram.total/1e9:.1f} GB  available={ram.available/1e9:.1f} GB')

GPU  : Tesla T4  total=15.6 GB  free=15.6 GB
RAM  : total=13.6 GB  available=12.1 GB


In [ ]:
# ── Cell 2: Select the training run and apply configuration ──────────────────
import sys
sys.path.insert(0, 'code/fine-tuning')
import train as t

# Pick which of the three thesis runs to execute (see PRESETS in train.py / D3.7):
#   'codebert-sqli'       — the product: CodeBERT on the FULL corpus (longest run)
#   'bert-sqli-500k'      — PP6 control: generic BERT, capped at 500K rows/split
#   'codebert-sqli-500k'  — PP6 control: CodeBERT at the same 500K budget
RUN = 'codebert-sqli'

t.apply_preset(RUN)

# Dataset on Google Drive (upload compiled_dataset.csv here before running).
# For a quick pipeline check, point this at the committed fixture instead:
#   t.DATASET_PATH = 'code/datasets/fixture/fixture_dataset.csv'
t.DATASET_PATH = '/content/drive/MyDrive/mestrado/compiled_dataset.csv'

# Model output on Google Drive, one folder per run (survives Colab session resets;
# checkpoint/resume state lives inside this folder, so each run resumes independently)
t.OUTPUT_DIR   = f'/content/drive/MyDrive/mestrado/models/{RUN}'

# Batch size — T4 (16 GB): 32 is safe; try 64 if no OOM; A100: 64–128
t.BATCH_SIZE   = 32

print(f'Run            : {RUN}')
print(f'Model          : {t.MODEL_NAME}')
print(f'Dataset        : {t.DATASET_PATH}')
print(f'Output         : {t.OUTPUT_DIR}')
print(f'Batch          : {t.BATCH_SIZE}')
print(f'Max rows/split : {t.MAX_ROWS_PER_SPLIT if t.MAX_ROWS_PER_SPLIT is None else format(t.MAX_ROWS_PER_SPLIT, ",")}')

In [ ]:
# ── Cell 3: Train ─────────────────────────────────────────────────────────────
t.main()

2026-05-09 20:19:29,903  INFO      Log file: code/fine-tuning/logs/train_20260509_201929.log
2026-05-09 20:19:29,904  INFO      Model:       microsoft/codebert-base
2026-05-09 20:19:29,905  INFO      Dataset:     /content/drive/MyDrive/mestrado/compiled_dataset.csv
2026-05-09 20:19:29,905  INFO      Smoke test:  False
2026-05-09 20:19:29,914  INFO      Device:      cuda
2026-05-09 20:19:29,915  INFO      GPU:         Tesla T4  14.6 GB
2026-05-09 20:19:29,916  INFO      Loading split='train' ...
2026-05-09 20:20:04,795  INFO        Capped to 300,000 rows (MAX_ROWS_PER_SPLIT)
2026-05-09 20:20:04,798  INFO        300,000 rows — benign=226,506  malicious=73,494
2026-05-09 20:20:05,286  INFO      Loading split='val' ...
2026-05-09 20:20:40,655  INFO        Capped to 300,000 rows (MAX_ROWS_PER_SPLIT)
2026-05-09 20:20:40,658  INFO        300,000 rows — benign=226,291  malicious=73,709
2026-05-09 20:20:41,045  INFO        After undersample: benign=146,988  malicious=73,494  ratio=2.0:1
2026-05

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-05-09 20:22:01,143  INFO      HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/commits/refs%2Fpr%2F9 "HTTP/1.1 200 OK"
2026-05-09 20:22:01,237  INFO      HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-09 20:22:01,240  WARNING   Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-09 20:22:01,341  INFO      HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors "HTTP/1.1 302 Found"
RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
clas

: 

In [ ]:
!nvidia-smi

In [ ]:
import os, glob
  logs = sorted(glob.glob('code/fine-tuning/logs/train_*.log'))
  print(logs)
  !tail -30 {logs[-1]}

In [ ]:
# ── Cell 4: Download run log ──────────────────────────────────────────────────
import glob
from google.colab import files

log_files = sorted(glob.glob('code/fine-tuning/logs/train_*.log'))
if log_files:
    latest = log_files[-1]
    print(f'Downloading: {latest}')
    files.download(latest)
else:
    print('No log files found in code/fine-tuning/logs/')